# Partie 2.2 — Approche profonde (CNN) — Template à compléter

**Pré-requis :** les cellules de la partie 2.1 ont été exécutées (on réutilise `ordre_images`, `labels`,
`idx_train`, `idx_val`, `idx_test`, `y_train`, `y_val`, `y_test`, `noms_classes`).
On garde **exactement le même découpage** qu'en 2.1 pour que la comparaison soit juste.

Les `___` sont à compléter, le commentaire `# 🔎` dit quoi chercher.

### Fonctions à connaître
| Fonction | Rôle |
|---|---|
| `keras.utils.load_img(chemin, target_size=(h, w))` | charge une image et la redimensionne |
| `keras.utils.img_to_array(img)` | image → tableau NumPy (h, w, 3) |
| `layers.Rescaling(facteur)` | multiplie les pixels (ex. 0..255 → 0..1) |
| `layers.Conv2D(nb_filtres, taille, activation, padding)` | couche de convolution |
| `layers.MaxPooling2D(2)` | divise la taille des cartes par 2 (max sur blocs 2×2) |
| `layers.Flatten()` | aplatit les cartes (h, w, c) en un vecteur |
| `layers.GlobalAveragePooling2D()` | moyenne de chaque carte → 1 valeur par filtre |
| `layers.RandomFlip / RandomRotation / RandomZoom` | augmentation de données (actives seulement à l'entraînement) |
| `model.summary()` | affiche les couches et le nombre de paramètres |
| `keras.applications.VGG16(weights="imagenet", include_top=False)` | réseau pré-entraîné sans ses couches de décision |
| `layer.trainable = False` | fige les poids d'une couche (ou d'un modèle entier) |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)

TAILLE = 128     # 256 est possible mais ~4x plus lent et plus gourmand en mémoire (sans GPU, reste à 128)
dossier = "Wang" # dossier contenant 0.jpg ... 999.jpg

## 1. Chargement des images
On charge les images **dans le même ordre que les descripteurs** (`ordre_images`) :
ainsi `idx_train`, `idx_val`, `idx_test` désignent les mêmes images qu'en 2.1.
On stocke en `uint8` (0..255) pour économiser la mémoire ; la mise à l'échelle se fera dans le modèle.

In [ ]:
images = []
for nom in ordre_images:
    img = keras.utils.load_img(os.path.join(dossier, nom), target_size=(nom.height,nom.width))   # 🔎 tuple (hauteur, largeur)
    images.append(keras.utils.img_to_array(img, dtype="uint8"))

X_img = np.stack(images)

# Vérification : les labels déduits des noms sont bien ceux de la partie 2.1
labels_img = np.array([___ for nom in ordre_images])        # 🔎 même formule qu'en 2.1.1
assert (labels_img == labels).all()

print(X_img.shape, X_img.dtype, f"{X_img.nbytes / 1e6:.0f} Mo")   # attendu : (1000, 128, 128, 3)

In [ ]:
# Une image par classe pour vérifier le chargement
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for c, ax in enumerate(axes.flat):
    i = np.where(labels == c)[0][0]
    ax.imshow(X_img[i])
    ax.set_title(noms_classes[c])
    ax.axis("off")
plt.show()

## 2. Découpage (mêmes indices qu'en 2.1)

In [ ]:
X_train_img = X_img[___]     # 🔎
X_val_img   = X_img[___]     # 🔎
X_test_img  = X_img[___]     # 🔎

print(X_train_img.shape, X_val_img.shape, X_test_img.shape)

## 3. Outils : entraînement et courbes

In [ ]:
def entrainer_cnn(model, X_tr, X_v, epochs=50, batch_size=32, patience=10, verbose=1):
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True)

    history = model.fit(
        X_tr, y_train,
        validation_data=(X_v, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=[early_stopping], verbose=verbose)

    pred = np.argmax(model.predict(X_v, verbose=0), axis=1)
    return {"model": model, "history": history,
            "acc": accuracy_score(y_val, pred), "pred_val": pred}


def tracer(history, titre=""):
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h["loss"], label="apprentissage"); ax1.plot(h["val_loss"], label="validation")
    ax1.set_title(f"Loss — {titre}"); ax1.set_xlabel("Époque"); ax1.legend(); ax1.grid(True)
    ax2.plot(h["accuracy"], label="apprentissage"); ax2.plot(h["val_accuracy"], label="validation")
    ax2.set_title(f"Accuracy — {titre}"); ax2.set_xlabel("Époque"); ax2.legend(); ax2.grid(True)
    plt.show()

resultats_cnn = {}

## 4. CNN simple (celui du sujet)
1 convolution + 1 pooling + décision directe. Regarde bien le **nombre de paramètres** dans `summary()` :
où sont-ils concentrés ?

In [ ]:
def cnn_simple():
    model = keras.Sequential()
    model.add(keras.Input(shape=(TAILLE, TAILLE, ___)))      # 🔎 nombre de canaux d'une image couleur
    model.add(layers.Rescaling(___))                          # 🔎 facteur pour passer de 0..255 à 0..1
    model.add(layers.Conv2D(32, (3, 3), activation="relu"))
    model.add(layers.MaxPooling2D(pool_size=___))             # 🔎 blocs 2x2
    model.add(layers.___())                                   # 🔎 cartes (h, w, c) -> vecteur
    model.add(layers.Dense(10, activation="softmax"))

    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

m = cnn_simple()
m.summary()

In [ ]:
resultats_cnn["simple"] = entrainer_cnn(cnn_simple(), X_train_img, X_val_img)
print(f"Accuracy validation : {resultats_cnn['simple']['acc']:.3f}")
tracer(resultats_cnn["simple"]["history"], "CNN simple")

## 5. Influence des paramètres
On empile des **blocs** convolution → pooling. Convention classique : on **double le nombre de filtres**
à chaque bloc (les cartes rétrécissent, on compense en en mettant plus).

In [ ]:
def cnn_blocs(nb_blocs=2, filtres=32, neurones_dense=64):
    model = keras.Sequential()
    model.add(keras.Input(shape=(TAILLE, TAILLE, 3)))
    model.add(layers.Rescaling(1 / 255))

    for b in range(nb_blocs):
        model.add(layers.Conv2D(___, 3, activation="relu", padding="same"))  # 🔎 filtres au bloc b (32, 64, 128...)
        model.add(layers.MaxPooling2D(2))

    model.add(layers.Flatten())
    model.add(layers.Dense(neurones_dense, activation="relu"))
    model.add(layers.Dense(10, activation="softmax"))

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
configs_cnn = {
    "1 bloc (32)":     {"nb_blocs": 1, "filtres": 32},
    "2 blocs (32-64)": {"nb_blocs": 2, "filtres": 32},
    "3 blocs (32-128)":{"nb_blocs": 3, "filtres": 32},
    "3 blocs (16-64)": {"nb_blocs": 3, "filtres": 16},
    # ajoute tes essais : neurones_dense, 4 blocs...
}

for nom, conf in configs_cnn.items():
    keras.utils.set_random_seed(42)
    model = cnn_blocs(**conf)
    resultats_cnn[nom] = entrainer_cnn(model, X_train_img, X_val_img, verbose=0)
    r = resultats_cnn[nom]
    print(f"{nom:<18} params={model.count_params():>10,}  acc val={r['acc']:.3f}  "
          f"({len(r['history'].history['loss'])} époques)")

In [ ]:
for nom in configs_cnn:
    tracer(resultats_cnn[nom]["history"], nom)

**À observer :** l'écart train / validation (surapprentissage ?), le lien entre nombre de paramètres et
résultat, la vitesse à laquelle la loss d'apprentissage tombe à 0.

## 6. Architecture plus complexe + augmentation de données
Trois ajouts contre le surapprentissage :
- **augmentation** : chaque époque, le réseau voit des versions légèrement modifiées des images ;
- **BatchNormalization** + **Dropout** ;
- **GlobalAveragePooling2D** à la place de `Flatten` (beaucoup moins de paramètres).

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("___"),          # 🔎 quel retournement garde une image réaliste ? (pas "vertical" : un bus à l'envers…)
    layers.RandomRotation(___),        # 🔎 fraction de tour, ex. 0.05 ≈ ±18°
    layers.RandomZoom(___),            # 🔎 ex. 0.1 = ±10 %
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# Visualiser l'effet : la même image, augmentée 8 fois
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax in axes:
    img_aug = augmentation(X_img[150:151].astype("float32"), training=True)
    ax.imshow(np.array(img_aug[0]).astype("uint8"))
    ax.axis("off")
plt.show()

In [ ]:
def cnn_complet(nb_blocs=4, filtres=32, dropout=0.5):
    model = keras.Sequential()
    model.add(keras.Input(shape=(TAILLE, TAILLE, 3)))
    model.add(augmentation)
    model.add(layers.Rescaling(1 / 255))

    for b in range(nb_blocs):
        model.add(layers.Conv2D(filtres * 2**b, 3, padding="same", use_bias=False))
        model.add(___)                                      # 🔎 normalisation par lots (avant l'activation)
        model.add(layers.Activation("relu"))
        model.add(layers.MaxPooling2D(2))

    model.add(layers.___())                                 # 🔎 remplace Flatten : 1 moyenne par carte
    model.add(layers.Dropout(___))                          # 🔎 taux de dropout
    model.add(layers.Dense(10, activation="softmax"))

    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

m = cnn_complet()
m.summary()

In [ ]:
keras.utils.set_random_seed(42)
# Avec l'augmentation, l'apprentissage est plus lent : plus d'époques et plus de patience
resultats_cnn["complet + augm."] = entrainer_cnn(cnn_complet(), X_train_img, X_val_img,
                                                 epochs=___, patience=___, verbose=1)   # 🔎 ex. 150 et 20
print(f"Accuracy validation : {resultats_cnn['complet + augm.']['acc']:.3f}")
tracer(resultats_cnn["complet + augm."]["history"], "CNN complet + augmentation")

## 7. Apprentissage par transfert (VGG16)
On réutilise les convolutions de VGG16, entraînées sur ImageNet (1,2 million d'images, 1000 classes),
et on ne réapprend que la décision. ⚠️ Les poids (~60 Mo) sont téléchargés au premier lancement.

VGG16 attend ses images préparées **comme pendant son entraînement** → on utilise sa propre
fonction `preprocess_input` (sur des pixels 0..255, sans `Rescaling`).

In [ ]:
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input

X_train_tl = preprocess_input(X_train_img.astype("float32"))
X_val_tl   = preprocess_input(X_val_img.astype("float32"))
X_test_tl  = ___                                            # 🔎 même chose pour le test

base = VGG16(weights="imagenet", include_top=___,           # 🔎 garder ou non ses couches de décision (1000 classes) ?
             input_shape=(TAILLE, TAILLE, 3))
base.trainable = ___                                        # 🔎 on fige les convolutions

entrees = keras.Input(shape=(TAILLE, TAILLE, 3))
x = augmentation(entrees)
x = base(x, training=False)          # training=False : la base reste en mode "inférence"
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
sorties = layers.Dense(___, activation="___")(x)            # 🔎 couche de décision
model_tl = keras.Model(entrees, sorties)

model_tl.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_tl.summary()      # compare "Trainable params" et "Non-trainable params"

In [ ]:
keras.utils.set_random_seed(42)
resultats_cnn["VGG16 transfert"] = entrainer_cnn(model_tl, X_train_tl, X_val_tl,
                                                 epochs=50, patience=10, verbose=1)
print(f"Accuracy validation : {resultats_cnn['VGG16 transfert']['acc']:.3f}")
tracer(resultats_cnn["VGG16 transfert"]["history"], "VGG16 (convolutions figées)")

### 7 bis (optionnel) — Fine-tuning
On dégèle le **dernier bloc** de VGG16 (`block5_*`) et on réentraîne avec un taux d'apprentissage
**très petit**, pour ajuster légèrement les filtres sans détruire ce qu'ils ont appris.

In [ ]:
base.trainable = True
for layer in base.layers:
    layer.trainable = layer.name.startswith("___")          # 🔎 nom du dernier bloc de VGG16

# Après avoir changé trainable, il FAUT recompiler
model_tl.compile(optimizer=keras.optimizers.Adam(___),       # 🔎 taux très petit, ex. 1e-5
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])

resultats_cnn["VGG16 fine-tuning"] = entrainer_cnn(model_tl, X_train_tl, X_val_tl,
                                                   epochs=30, patience=8, verbose=1)
print(f"Accuracy validation : {resultats_cnn['VGG16 fine-tuning']['acc']:.3f}")
tracer(resultats_cnn["VGG16 fine-tuning"]["history"], "VGG16 fine-tuning")

## 8. Récapitulatif sur la validation

In [ ]:
pd.DataFrame({
    "Modèle": list(resultats_cnn.keys()),
    "Params entraînables": [sum(np.prod(w.shape) for w in r["model"].trainable_weights)
                            for r in resultats_cnn.values()],
    "Acc val": [r["acc"] for r in resultats_cnn.values()],
    "Époques": [len(r["history"].history["loss"]) for r in resultats_cnn.values()],
}).sort_values("Acc val", ascending=False)

## 9. Évaluation finale sur le TEST et comparaison avec la partie 2.1
⚠️ Pour les modèles VGG16, il faut donner `X_test_tl` (images prétraitées), pas `X_test_img`.
Note : `model_tl` a été modifié par le fine-tuning — « VGG16 transfert » et « VGG16 fine-tuning »
pointent vers le même objet. Si tu veux évaluer les deux, évalue « transfert » avant la section 7 bis.

In [ ]:
modeles_test = {
    "CNN simple":            (resultats_cnn["simple"]["model"],           X_test_img),
    "CNN complet + augm.":   (resultats_cnn["complet + augm."]["model"],  X_test_img),
    "VGG16 (fine-tuning)":   (model_tl,                                   ___),       # 🔎 quelles données ?
}

lignes = []
for nom, (model, X_t) in modeles_test.items():
    pred = ___                                              # 🔎 predict + argmax
    lignes.append({"Modèle": nom, "Acc test": accuracy_score(y_test, pred)})

# Rappel des meilleurs résultats de la partie 2.1 (descripteurs)
lignes.append({"Modèle": "MLP descripteurs (Complet, dropout 0.5)", "Acc test": acc_test_mlp})
lignes.append({"Modèle": "KppV descripteurs (Complet)", "Acc test": resultats_knn["Complet"]["acc_test"]})

pd.DataFrame(lignes).sort_values("Acc test", ascending=False)

In [ ]:
# Matrice de confusion du meilleur modèle profond
pred_best = np.argmax(model_tl.predict(X_test_tl, verbose=0), axis=1)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_best), display_labels=noms_classes).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title("Matrice de confusion (test) — VGG16")
plt.show()

## Questions pour le rapport
- CNN simple : combien de paramètres, où sont-ils ? Surapprentissage ? Pourquoi ?
- Effet du nombre de blocs / filtres sur la validation et sur le nombre de paramètres.
- L'augmentation de données réduit-elle l'écart train/validation ? À quel prix (nombre d'époques) ?
- CNN entraînés de zéro vs descripteurs manuels de 2.1 : lequel gagne avec seulement 640 images ?
- Pourquoi le transfert depuis ImageNet marche-t-il si bien ici ? Les confusions (Plages/Montagnes,
  Afrique/Éléphants) persistent-elles ?